# 📰 ESPN News API - Bronze Layer Ingestion

## 🎯 Purpose

Ingest rich player news content from ESPN's unofficial News API into `main.fantasai.bronze_player_news_espn_api`.

---

## 🔗 Data Source

**API:** ESPN Site API v2  
**Endpoint:** `https://site.api.espn.com/apis/site/v2/sports/football/nfl/news?player={espn_id}`  
**Cost:** ✅ FREE (no API key required)  
**Rate Limits:** ✅ None observed  
**Authentication:** ✅ None required  

---

## 📊 Data Fields

* `article_id` - ESPN article ID (unique)
* `player_id` - Master player ID (from gold_player_dim)
* `espn_player_id` - ESPN player ID
* `player_name` - Player name
* `headline` - Article headline
* `description` - Article description/summary
* `article_type` - Type (Story, Media, etc.)
* `published_at` - Publication timestamp
* `last_modified` - Last modification timestamp
* `article_url` - Link to full article
* `image_url` - Featured image URL
* `categories` - Array of categories (teams, players, etc.)
* `fetched_at` - Ingestion timestamp

---

## 🏗️ Architecture

```
[ESPN News API]
      ↓
[Fetch for all players with ESPN IDs]
      ↓
[main.fantasai.bronze_player_news_espn_api]  ← Raw ESPN news articles
      ↓
[silver_player_news_unified]  ← Deduplicated across all sources
      ↓
[gold_player_news]  ← Concatenated by player
```

---

## ⚙️ Execution

**Schedule:** Daily at 7:00 AM UTC (with ESPN ingestion)  
**Runtime:** ~5-10 minutes for 4,640 players  
**Dependencies:** Requires `gold_player_dim` with ESPN IDs  
**Deduplication:** Skips articles already in bronze table  

---

## 📝 Notes

* Not all players have ESPN news (typically only stars/starters)
* API returns up to 50 most recent articles per player
* Articles can mention multiple players (stored per player)
* Unofficial API (no SLA, but very stable)

In [0]:
# =============================================================================
# ESPN NEWS API INGESTION - CONFIGURATION
# =============================================================================

from datetime import datetime
import time

print("="*80)
print("📰 ESPN News API Ingestion Configuration")
print("="*80)

# === MODE SELECTION ===
MODE = "production"  # Options: "test" (10 players), "production" (all players)

# === API CONFIGURATION ===
ESPN_NEWS_API_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/news"
REQUEST_TIMEOUT = 10  # seconds
RATE_LIMIT_DELAY = 0.1  # seconds between requests (be polite)

# === TABLE CONFIGURATION ===
BRONZE_TABLE = "main.fantasai.bronze_player_news_espn_api"
PLAYER_DIM_TABLE = "main.fantasai.gold_player_dim"

# === EXECUTION SETTINGS ===
if MODE == "test":
    PLAYER_LIMIT = 10
    print("\n🧪 TEST MODE - Processing 10 players")
else:
    PLAYER_LIMIT = None
    print("\n⚡ PRODUCTION MODE - Processing all players with ESPN IDs")

print(f"\n📋 Configuration:")
print(f"   API Endpoint: {ESPN_NEWS_API_URL}")
print(f"   Bronze Table: {BRONZE_TABLE}")
print(f"   Source Table: {PLAYER_DIM_TABLE}")
print(f"   Rate Limit: {RATE_LIMIT_DELAY}s between requests")
print(f"   Request Timeout: {REQUEST_TIMEOUT}s")

if PLAYER_LIMIT:
    print(f"   Player Limit: {PLAYER_LIMIT} (test mode)")

print("\n" + "="*80)

In [0]:
# Install requests library for HTTP calls
%pip install requests --quiet

print("✅ Dependencies installed")

In [0]:
%sql
-- Get all players with ESPN IDs from gold player ID mapping
-- This table maps master_player_id to source-specific IDs

CREATE OR REPLACE TEMP VIEW players_with_espn_ids AS
SELECT 
  m.master_player_id,
  m.source_player_name as player_name,
  m.position,
  m.team,
  m.source_player_id as espn_id
FROM main.fantasai.gold_player_id_mapping m
WHERE m.source = 'espn_public'
  AND m.source_player_id IS NOT NULL
  AND m.source_player_id != ''
  AND m.source_player_id != '0'
ORDER BY m.source_player_name;

-- Show sample
SELECT 
  COUNT(*) as total_players,
  COUNT(DISTINCT position) as positions,
  COUNT(DISTINCT team) as teams
FROM players_with_espn_ids;

In [0]:
# =============================================================================
# FETCH ESPN NEWS ARTICLES FOR ALL PLAYERS
# =============================================================================

import requests
import json
from datetime import datetime
import time
from typing import List, Dict, Optional

print("="*80)
print("📡 Fetching ESPN News Articles")
print("="*80)

# Get player list from temp view
players_df = spark.table("players_with_espn_ids").toPandas()

if PLAYER_LIMIT:
    players_df = players_df.head(PLAYER_LIMIT)

print(f"\n📊 Processing {len(players_df)} players with ESPN IDs")
print(f"\n⏱️  Estimated time: ~{len(players_df) * RATE_LIMIT_DELAY / 60:.1f} minutes\n")

# === FETCH FUNCTION ===
def fetch_espn_news(espn_id: str, timeout: int = REQUEST_TIMEOUT) -> Optional[Dict]:
    """Fetch news articles for a player from ESPN API."""
    try:
        url = f"{ESPN_NEWS_API_URL}?player={espn_id}"
        response = requests.get(url, timeout=timeout)
        
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except Exception as e:
        return None

# === PROCESS ALL PLAYERS ===
all_articles = []
players_with_news = 0
total_articles = 0
errors = 0

for idx, row in players_df.iterrows():
    espn_id = row['espn_id']
    master_player_id = row['master_player_id']
    player_name = row['player_name']
    
    # Progress indicator every 100 players
    if (idx + 1) % 100 == 0:
        print(f"   ✓ Processed {idx + 1}/{len(players_df)} players ({players_with_news} with news, {total_articles} articles)")
    
    # Fetch news
    data = fetch_espn_news(espn_id)
    
    if data and 'articles' in data:
        articles = data.get('articles', [])
        
        if articles:
            players_with_news += 1
            
            # Parse each article
            for article in articles:
                try:
                    # Extract image URL (first image if available)
                    images = article.get('images', [])
                    image_url = images[0].get('url') if images else None
                    
                    # Extract article URL (first link if available)
                    links = article.get('links', {})
                    article_url = None
                    if 'web' in links and 'href' in links['web']:
                        article_url = links['web']['href']
                    
                    # Extract categories (teams, players, etc.)
                    categories = article.get('categories', [])
                    category_list = [cat.get('description', '') for cat in categories if 'description' in cat]
                    
                    # Build article record
                    article_record = {
                        'article_id': str(article.get('id')),
                        'player_id': master_player_id,
                        'espn_player_id': espn_id,
                        'player_name': player_name,
                        'headline': article.get('headline', ''),
                        'description': article.get('description', ''),
                        'article_type': article.get('type', ''),
                        'published_at': article.get('published', ''),
                        'last_modified': article.get('lastModified', ''),
                        'article_url': article_url,
                        'image_url': image_url,
                        'categories': category_list,
                        'fetched_at': datetime.utcnow().isoformat() + 'Z'
                    }
                    
                    all_articles.append(article_record)
                    total_articles += 1
                    
                except Exception as e:
                    errors += 1
                    continue
    
    # Rate limiting (be polite to ESPN)
    time.sleep(RATE_LIMIT_DELAY)

print("\n" + "="*80)
print("\n📊 Fetch Results:")
print(f"   Players Processed: {len(players_df)}")
print(f"   Players with News: {players_with_news} ({players_with_news/len(players_df)*100:.1f}%)")
print(f"   Total Articles: {total_articles}")
print(f"   Errors: {errors}")
print(f"   Avg Articles per Player (with news): {total_articles/players_with_news:.1f}")

if total_articles == 0:
    print("\n⚠️  No articles found. Check API endpoint or ESPN IDs.")
else:
    print(f"\n✅ Successfully fetched {total_articles} articles")

In [0]:
%sql
-- Create bronze table for ESPN news articles (if not exists)
-- This table stores raw ESPN news articles with player associations

CREATE TABLE IF NOT EXISTS main.fantasai.bronze_player_news_espn_api (
  article_id STRING NOT NULL COMMENT 'ESPN article unique ID',
  player_id STRING NOT NULL COMMENT 'Master player ID from gold_player_dim',
  espn_player_id STRING COMMENT 'ESPN player ID used for API call',
  player_name STRING COMMENT 'Player name for reference',
  headline STRING COMMENT 'Article headline',
  description STRING COMMENT 'Article description/summary',
  article_type STRING COMMENT 'Article type (Story, Media, etc.)',
  published_at TIMESTAMP COMMENT 'Original publication timestamp',
  last_modified TIMESTAMP COMMENT 'Last modification timestamp',
  article_url STRING COMMENT 'URL to full article',
  image_url STRING COMMENT 'Featured image URL',
  categories ARRAY<STRING> COMMENT 'Article categories (teams, leagues, etc.)',
  fetched_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp',
  CONSTRAINT pk_espn_news PRIMARY KEY (article_id, player_id)
)
COMMENT 'Raw ESPN news articles from ESPN Site API v2 - Bronze layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

DESCRIBE EXTENDED main.fantasai.bronze_player_news_espn_api;

In [0]:
# =============================================================================
# WRITE TO BRONZE TABLE WITH DEDUPLICATION
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType

print("="*80)
print("💾 Writing Articles to Bronze Table")
print("="*80)

if total_articles == 0:
    print("\n⚠️  No articles to write. Skipping write operation.")
else:
    # Convert to Spark DataFrame
    articles_df = spark.createDataFrame(all_articles)
    
    # Convert timestamp strings to proper timestamps
    articles_df = articles_df \
        .withColumn('published_at', F.to_timestamp('published_at')) \
        .withColumn('last_modified', F.to_timestamp('last_modified')) \
        .withColumn('fetched_at', F.to_timestamp('fetched_at'))
    
    print(f"\n📊 Prepared {articles_df.count()} articles for insertion")
    
    # Check for existing articles (deduplication)
    existing_articles_df = spark.sql(f"""
        SELECT DISTINCT article_id, player_id
        FROM {BRONZE_TABLE}
    """)
    
    existing_count = existing_articles_df.count()
    print(f"📋 Found {existing_count} existing articles in database")
    
    # Left anti join to find new articles only
    new_articles_df = articles_df.join(
        existing_articles_df,
        on=['article_id', 'player_id'],
        how='left_anti'
    )
    
    new_count = new_articles_df.count()
    duplicate_count = articles_df.count() - new_count
    
    print(f"\n🔍 Deduplication results:")
    print(f"   New articles: {new_count}")
    print(f"   Duplicates skipped: {duplicate_count}")
    
    if new_count > 0:
        print(f"\n💾 Writing {new_count} new articles to {BRONZE_TABLE}...")
        
        new_articles_df.write \
            .mode('append') \
            .saveAsTable(BRONZE_TABLE)
        
        print("\n✅ Write complete!")
        
        # Show sample of new articles
        print("\n📰 Sample of new articles:")
        new_articles_df.select(
            'player_name',
            F.substring('headline', 1, 60).alias('headline_preview'),
            'article_type',
            'published_at'
        ).orderBy(F.desc('published_at')).show(10, truncate=False)
    else:
        print("\n✓ No new articles to write (all duplicates)")

print("\n" + "="*80)

In [0]:
%sql
-- Validate ingestion: Show article counts by player

SELECT 
  player_name,
  COUNT(*) as article_count,
  MAX(published_at) as latest_article,
  MIN(published_at) as oldest_article,
  COUNT(DISTINCT article_type) as article_types,
  MAX(fetched_at) as last_fetched
FROM main.fantasai.bronze_player_news_espn_api
GROUP BY player_name
ORDER BY article_count DESC
LIMIT 20;

In [0]:
%sql
-- Show most recent ESPN articles across all players

SELECT 
  player_name,
  SUBSTRING(headline, 1, 80) as headline_preview,
  article_type,
  published_at,
  DATEDIFF(HOUR, published_at, CURRENT_TIMESTAMP()) as hours_ago,
  article_url
FROM main.fantasai.bronze_player_news_espn_api
ORDER BY published_at DESC
LIMIT 25;

In [0]:
%sql
-- Overall summary statistics for ESPN news table

SELECT 
  COUNT(*) as total_articles,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(DISTINCT article_id) as unique_articles,
  COUNT(DISTINCT article_type) as article_types,
  MIN(published_at) as oldest_article,
  MAX(published_at) as newest_article,
  MAX(fetched_at) as last_ingestion_run,
  ROUND(AVG(LENGTH(headline)), 0) as avg_headline_length,
  ROUND(AVG(LENGTH(description)), 0) as avg_description_length,
  SUM(CASE WHEN image_url IS NOT NULL THEN 1 ELSE 0 END) as articles_with_images,
  SUM(CASE WHEN article_url IS NOT NULL THEN 1 ELSE 0 END) as articles_with_urls
FROM main.fantasai.bronze_player_news_espn_api;